# Lesson 16 · The attention mechanism: reading Vietnamese numbers by looking back at the right place

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/16-attention-en.ipynb)

This notebook accompanies [Lesson 16 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/attention/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/16-attention.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
The task is a small “translation” that a machine can check exactly: read how a number is read aloud in Vietnamese
(“hai nghìn không trăm hai mươi sáu”, word for word “two thousand zero hundred two ten six”) and write its digits back
(2026). The Vietnamese readings are the models' data, so they stay Vietnamese here; section 2 explains the reading
rules. Using numpy, we run the very trained models the demo uses: a GRU encoder–decoder **without** attention, and
the same model **with** Bahdanau's attention.

**What you'll do:**
1. Write a reader for Vietnamese numbers and generate the test set, then run the model without attention: where it
   misreads as the sentences get longer.
2. Compute one attention step by hand: scores, softmax, weighted average; draw the attention map between the two
   sentences.
3. Intervene on the attention weights to see whether they really “explain” the prediction.
4. Measure why the Transformer divides $q \cdot k$ by $\sqrt{d_k}$, on random vectors and on real GRU states.
5. Replace the encoder with a self-attention layer and see that it doesn't know word order without position vectors.

**Background:** [Lesson 15 · RNN & LSTM](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn/) (the GRU and its
gates) and [Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/).
**Run time:** under a minute on a CPU, including the first download of the two results files (~0.6 MB).

Every number in the notebook is computed by the cells below (or read from the results files the demo uses), and
matches the numbers in the reel and the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import hashlib
import itertools
import json
import math
import urllib.request
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
WEIGHTS = LinearSegmentedColormap.from_list("white_blue", ["#ffffff", BLUE])  # attention weights: 0 → 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never prints “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is
    long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(num(53259, 0), num(0.7129, 2), pct(0.7233))

## 2 · The data: reading a number in Vietnamese

The source sentence is the standard Northern Vietnamese reading of an integer with 1–12 digits; the target sentence
is its digits. The digits 0–9 are “không, một, hai, ba, bốn, năm, sáu, bảy, tám, chín”; “trăm” is hundred, and
“nghìn”, “triệu”, “tỷ” are thousand, million, billion. The reading rules include all the tricky parts of Vietnamese:
- **mươi / mười:** 20 is “hai mươi” (two tens), 10 is “mười” (ten);
- **mốt, tư, lăm:** 21 is “hai mươi mốt”, 24 is “hai mươi tư”, 15 is “mười lăm”, 25 is “hai mươi lăm” (after “mươi”,
  the digits 1, 4 and 5 are read “mốt”, “tư”, “lăm” instead of “một”, “bốn”, “năm”; after “mười”, 5 is read “lăm”);
- **linh:** 105 is “một trăm linh năm” (“linh” marks the empty tens place: one hundred, no tens, five);
- **không trăm:** a three-digit group in the middle is always read with its hundreds (2026 is
  “hai nghìn **không trăm** hai mươi sáu”, two thousand **zero hundred** twenty-six);
- **dropping 000 groups:** 2,000,000 is just “hai triệu” (two million).

The whole vocabulary has only 20 words. The cell below writes the reader (identical to the course's
`reel/attention/numvi.py`) and tests it on a few numbers.

In [ ]:
D = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
SCALE = ["", "nghìn", "triệu", "tỷ"]


def tens_units(b, c):
    """The tens digit b and the units digit c of a three-digit group."""
    if b == 0:
        return ["linh", D[c]] if c else []
    if b == 1:
        return ["mười"] + (["lăm" if c == 5 else D[c]] if c else [])
    return [D[b], "mươi"] + ([{1: "mốt", 4: "tư", 5: "lăm"}.get(c, D[c])] if c else [])


def group(g, lead):
    a, b, c = g // 100, g // 10 % 10, g % 10
    if lead and a == 0:  # the first group doesn't read a leading 0: 26 is "hai mươi sáu", not "không trăm…"
        return [D[c]] if b == 0 else tens_units(b, c)
    return [D[a], "trăm"] + tens_units(b, c)


def read(n):
    """Integer n ≥ 0 → list of words."""
    if n == 0:
        return ["không"]
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    words = []
    for k in reversed(range(len(groups))):
        lead = k == len(groups) - 1
        if groups[k] == 0 and not lead:  # skip the 000 groups in the middle
            continue
        words += group(groups[k], lead)
        if SCALE[k]:
            words.append(SCALE[k])
    return words

Test the reader on numbers that hit every tricky rule (mốt, tư, lăm, linh, không trăm, dropping 000 groups), then
count the vocabulary.

In [ ]:
CHECKS = {
    15: "mười lăm",
    21: "hai mươi mốt",
    24: "hai mươi tư",
    105: "một trăm linh năm",
    2026: "hai nghìn không trăm hai mươi sáu",
    2000000: "hai triệu",
    1000005: "một triệu không trăm linh năm",
    1001000: "một triệu không trăm linh một nghìn",
}
for n, text in CHECKS.items():
    print(f"{num(n, 0):>10}  {' '.join(read(n))}")
    assert " ".join(read(n)) == text, "differs from the numvi.py of the reel and the demo"

vocab = {w for n in range(1000) for w in read(n)} | {"nghìn", "triệu", "tỷ"}  # 0–999 already use every word of a group
print(f"{len(vocab)} words:", " · ".join(sorted(vocab)))
assert len(vocab) == 20

Each random number has exactly $d$ digits. If the digits were uniformly random, a 000 group would have a probability
of only $0.001$, and the model would never learn to read round numbers. So the course's sampler makes each digit 0
with probability 0.25, and each three-digit group below the first one 000 with probability 0.2.

The test set is 300 numbers for each length $d = 1 \dots 12$, drawn from `default_rng(2000 + d)`: exactly the test
set of `reel/attention/train.py` (the validation set uses `default_rng(1000 + d)` to choose $\eta$; the model trains
on freshly drawn numbers at every step).

In [ ]:
P_ZERO, P_GROUP, N_TEST, D_MAX = 0.25, 0.2, 300, 12
SRC = ["<pad>", *D, "mốt", "tư", "lăm", "mười", "mươi", "trăm", "linh", "nghìn", "triệu", "tỷ"]  # id 0 is the padding
W2I = {w: i for i, w in enumerate(SRC)}
EOS, BOS = 10, 11  # outputs: 0–9 are the digits, 10 is ⟨end⟩; the decoder starts from symbol 11


def sample(rng, d, n, p0=P_ZERO, pg=P_GROUP):
    """n numbers with exactly d digits; each row holds the digits."""
    digs = np.where(rng.random((n, d)) < p0, 0, rng.integers(1, 10, (n, d)))
    digs[:, 0] = rng.integers(1, 10, n)  # the first digit is nonzero
    for g0 in range(d - 3, 0, -3):  # the three-digit groups counted from the right, except the first group
        digs[rng.random(n) < pg, g0 : g0 + 3] = 0
    return digs


def encode_set(digs):
    """Rows of digits → source word ids X (padded with 0), mask M (1 = a word), targets Y (the digits, then ⟨end⟩)."""
    srcs = [[W2I[w] for w in read(int("".join(map(str, r))))] for r in digs]
    X = np.zeros((len(srcs), max(map(len, srcs))), int)
    for i, s in enumerate(srcs):
        X[i, : len(s)] = s
    return X, (X > 0).astype(float), np.concatenate([digs, np.full((len(srcs), 1), EOS)], 1)


def as_int(row):
    return int("".join(map(str, row)))


TEST = {d: encode_set(sample(np.random.default_rng(2000 + d), d, N_TEST)) for d in range(1, D_MAX + 1)}
for d in (4, 7, 12):
    n = as_int(TEST[d][2][0, :-1])
    print(f"d = {d:>2}: {num(n, 0):>17} → {' '.join(read(n))}")
X12, M12, Y12 = TEST[12]
# the fraction of zero digits at each length d (single digits, not 000 groups)
has000 = [(TEST[d][2][:, 1:-1].reshape(-1) == 0).mean() for d in range(2, D_MAX + 1)]
print(f"fraction of zero digits (excluding the first digit): {pct(np.mean(has000), 1)}")
NB = as_int(Y12[90, :-1])  # the reel's example: row 90 of the 12-digit set
print(f"number 91 of the 12-digit set (row 90, counting from 0): {num(NB, 0)}, read with {len(read(NB))} words")
assert NB == 964131195884 and len(read(NB)) == 23, "differs from the reel: check the sampler"

The source sentence gets longer with the number of digits: a 12-digit number takes up to 23 words. The figure below
plots the mean number of words (blue line, round dots) and the shortest–longest range (gray band) of the 300 test
sentences at each length.

In [ ]:
lens = {d: TEST[d][1].sum(1) for d in TEST}
ds = np.arange(1, D_MAX + 1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.fill_between(
    ds, [lens[d].min() for d in ds], [lens[d].max() for d in ds], color=GREY, alpha=0.25, label="shortest–longest"
)
ax.plot(ds, [lens[d].mean() for d in ds], "o-", color=BLUE, label="mean")
ax.set(
    xlabel="number of digits d",
    ylabel="words in the source sentence",
    xticks=ds,
    title="The source sentence gets longer with the number of digits",
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()
print("longest:", int(max(lens[d].max() for d in ds)), "words")

The models were trained beforehand (6,000 Adam steps, each on 64 new numbers; too long for a notebook).
The cell below downloads exactly the two files the demo uses and checks their MD5 hashes:
- `attention-weights.json`: the float16 weights of seed 0, H = 64, for four models: no attention, additive
  attention, self-attention without positions and with positions;
- `attention-s2s.json`: the test results of the whole sweep (17 configurations × 3 values of $\eta$ × 3 seeds), used
  in sections 3, 7 and 8.

In [ ]:
# Results of a long training run (the same files as the demo on the web), downloaded from the data/ folder of the
# public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


shipped = fetch_json("attention-weights.json", "fd5de1ef245d385fbbabedc5c5f2c10d")  # the demo's weights
sweep = fetch_json("attention-s2s.json", "1a66caffd83e34411148c3b2bb3c0d3a")  # the demo's sweep results
size = sum((Path("data") / n).stat().st_size for n in ("attention-weights.json", "attention-s2s.json"))
print(
    "models:",
    list(shipped["models"]),
    "· runs in the sweep file:",
    len(sweep["runs"]),
    f"· {num(size / 1e6, 1)} MB",
)

## 3 · Cramming a whole sentence into one vector

The encoder is a GRU (Lesson 15) that reads one word at a time; each word updates the hidden state $h_j$. The decoder
is another GRU that writes one digit at a time. In the model **without attention** (the RNN encoder–decoder of Cho et
al., 2014), the decoder sees only the final state $h_T$: it is the decoder's initial state and also the “context” $c$
at every step. Everything about the 23 words has to squeeze through exactly those 64 numbers.

The cell below rewrites the two networks in numpy, in the row convention $z = xW + b$, as `reel/attention/s2s.py`
does:
- `Ex[x]` is the row of the word $x$ in the embedding table, already multiplied by the input weights of all three
  parts of the GRU (since $x$ is a one-hot vector, $xW$ is just one row of $W$);
- the decoder takes the digit it just wrote, `Ey[y]`, together with the context $c$, and computes the logits
  $s_i W_o + c_i W_{co} + b_o$ for the 11 outputs (10 digits and ⟨end⟩).

In [ ]:
def unpack(model):
    """base64-encoded float16 → float64 arrays, as in the demo."""
    return {
        k: np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float64).reshape(v["shape"])
        for k, v in model.items()
    }


MODELS = {name: unpack(m) for name, m in shipped["models"].items()}
assert shipped["src"] == SRC


def sig(v):
    return 0.5 * (1 + np.tanh(0.5 * v))  # = 1 / (1 + e^(−v)), never overflows


def gru(ax, h, U):
    """One GRU step. ax = xW + b of the three parts (reset gate r, update gate z, candidate n), joined by column."""
    H = h.shape[1]
    rz = sig(ax[:, : 2 * H] + h @ U[:, : 2 * H])
    r, z = rz[:, :H], rz[:, H:]
    n = np.tanh(ax[:, 2 * H :] + (r * h) @ U[:, 2 * H :])
    return z * h + (1 - z) * n  # z near 1: keep the old state (the convention of Cho et al., as in Lesson 15)


def encode(p, X, M):
    """The GRU encoder: the state after each word Hs (B, S, H) and the final state h_T. Padding (M = 0) keeps h."""
    h = np.zeros((X.shape[0], p["Ue"].shape[0]))
    Hs = np.zeros((*X.shape, h.shape[1]))
    for j in range(X.shape[1]):
        m = M[:, j : j + 1]
        h = m * gru(p["Ex"][X[:, j]] + p["be"], h, p["Ue"]) + (1 - m) * h
        Hs[:, j] = h
    return Hs, h


for name, p in MODELS.items():
    print(f"{name:>5}: {num(sum(v.size for v in p.values()), 0):>6} parameters")
assert [sum(v.size for v in p.values()) for p in MODELS.values()] == [45003, 53259, 50379, 51915]

The decoder writes **greedily**: at each step it picks the output with the largest logit, then feeds that digit into
the next step. The function `decode` below serves both models; the attention part (`att="add"`) is explained in
section 4; for now you only need to know that with `att="none"` the context is always $c = h_T$.

A number counts as read correctly when **every** digit and the ⟨end⟩ symbol are all right (exact match, as in the
demo).

In [ ]:
def softmax_masked(e, M):
    """Row-wise softmax, ignoring the padding (M = 0)."""
    e = np.where(M > 0, e, -np.inf)
    a = np.exp(e - e.max(-1, keepdims=True))
    return a / a.sum(-1, keepdims=True)


def additive_scores(p, s, keys):
    """Bahdanau's scores: e_j = tanh(s W_a + h_j U_a) · v_a for every position j. keys = Hs @ U_a, precomputed."""
    return np.tanh(keys + (s @ p["Wa"])[:, None, :]) @ p["va"]


def decode(p, Hs, s, M, steps, att, edit=None):
    """Greedy decoding. Returns the outputs (B, steps) and the attention weights (B, steps, S) if there is attention.
    edit(a, M) lets you modify the attention weights before they are used (section 6)."""
    y, h_T = np.full(Hs.shape[0], BOS), s
    keys = Hs @ p["Ua"] if att == "add" else None  # h_j U_a doesn't depend on the step i: compute it once
    out, A = [], []
    for _ in range(steps):
        if att == "none":
            c = h_T  # the bottleneck: the same vector at every step
        else:
            a = softmax_masked(additive_scores(p, s, keys), M)
            a = edit(a, M) if edit else a
            A.append(a)
            c = np.einsum("bs,bsh->bh", a, Hs)  # weighted average of the h_j
        s = gru(p["Ey"][y] + c @ p["Wc"] + p["bd"], s, p["Ud"])
        y = (s @ p["Wo"] + c @ p["Wco"] + p["bo"]).argmax(1)
        out.append(y)
    return np.stack(out, 1), (np.stack(A, 1) if A else None)


def run(name, X, M, steps, edit=None):
    p = MODELS[name]
    Hs, h_T = encode(p, X, M)
    return decode(p, Hs, h_T, M, steps, "none" if name == "none" else "add", edit)


def digits_of(row):
    """The outputs before ⟨end⟩, joined into a string of digits."""
    row = [int(v) for v in row]
    return "".join(map(str, row[: row.index(EOS)] if EOS in row else row))

Run the model without attention on all 3,600 test numbers (300 per length). The cell also compares the counts of
correct readings with the seed-0 results the sweep stored, to make sure that this numpy version is exactly the
demo's model.

In [ ]:
def exact(name, d, edit=None):
    X, M, Y = TEST[d]
    return (run(name, X, M, Y.shape[1], edit)[0] == Y).all(1)


ok_none = {d: exact("none", d) for d in TEST}
print("no attention, numbers read correctly out of 300 at each length:")
print("  ".join(f"d{d}: {ok_none[d].sum()}" for d in TEST))
print(f"12 digits: {pct(ok_none[12].mean())}")


def stored(att, H=64, steps=6000):
    """One configuration in the demo's sweep file."""
    return next(r for r in sweep["runs"] if r["att"] == att and r["H"] == H and r["steps"] == steps)


def span(vals, f=pct):
    """The lowest–highest range across seeds."""
    lo, hi = min(vals), max(vals)
    return f(lo) if f(lo) == f(hi) else f"{f(lo)}–{f(hi)}"


def seed0_counts(att):
    return [round(N_TEST * v) for v in stored(att)["seeds"][0]["test"]["em"]]


assert [int(ok_none[d].sum()) for d in TEST] == seed0_counts("none"), "differs from the demo: check the model"

Up to 8 digits, the fixed vector makes almost no mistakes; from 9 digits on, the errors grow quickly, and at 12
digits (sentences of up to 23 words) only 72.33% of the numbers are read correctly. This is exactly what Cho et al.
(2014) saw in machine translation: quality drops quickly as sentences get longer.

### 🤔 Predict first
The model without attention misreads 83 of the 12-digit numbers. Where do you think the errors mostly are: **which
digits** (the first, the middle or the last ones)? And what **kind** of error is typical: one wrong digit, a missing
digit, or two digits swapped?

In [ ]:
out12, _ = run("none", X12, M12, 13)
wrong = np.flatnonzero(~ok_none[12])
per_pos = (out12[wrong] != Y12[wrong]).sum(0)[:12]
swaps = 0
for b in wrong:
    diff = np.flatnonzero(out12[b] != Y12[b])
    if len(diff) == 2 and diff[1] == diff[0] + 1 and (out12[b, diff] == Y12[b, diff[::-1]]).all():
        swaps += 1
print(f"{len(wrong)} wrong numbers; errors at digits 1…12:", per_pos.tolist())
print(f"{swaps} of the {len(wrong)} wrong numbers are wrong only because two adjacent digits are swapped")
assert len(wrong) == 83

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(np.arange(1, 13), per_pos, color=ORANGE, hatch="//", edgecolor="white")
ax.set(xlabel="digit position (1 = the first digit)", ylabel="errors", xticks=np.arange(1, 13))
ax.set_title("No attention, 12 digits: where the wrong digits are")
fmt_axes(fig)
plt.show()

The errors pile up in the **middle** of the number (digits 5–10), and most of them are two adjacent digits swapped:
the model still remembers *which digits* there are, but the 64 numbers of $h_T$ can't hold them in the right
*order*. The first two digits are almost never wrong, and the last digits are wrong less often than the middle ones:
those are the easiest parts to keep, the beginning and the part the GRU has just finished reading.

The reel's example is the test number 964,131,195,884 (23 words):

In [ ]:
words_b = read(NB)
Xb = np.array([[W2I[w] for w in words_b]])
Mb = np.ones(Xb.shape)
read_none, read_add = (digits_of(run(name, Xb, Mb, 14)[0][0]) for name in ("none", "add"))
wrong_at = [k for k, (o, t) in enumerate(zip(read_none, str(NB))) if o != t]
print(" ".join(words_b))
print(f"no attention:   {num(int(read_none), 0)}   (wrong at digits {[k + 1 for k in wrong_at]})")
print(f"with attention: {num(int(read_add), 0)}")
assert read_none == "964131915884" and wrong_at == [6, 7] and read_add == str(NB), "differs from the reel"

“195” became “915”: exactly the swap above. The model with attention reads that number correctly (sections 4–5 look
at how it does it).

But is the fixed vector a real **wall**, or does it only make learning harder? The course's sweep answers that. The
figure below plots the fraction read correctly against the number of digits, at H = 64, for 3 seeds (line: median,
band: lowest–highest). Orange, squares: no attention after 6,000 steps; green, triangles, dashed: the same model
trained 4 times longer (24,000 steps); blue, round dots: with attention after 6,000 steps.

In [ ]:
def em_seeds(att, H=64, steps=6000):
    """Fraction read correctly at d = 1…12 for the 3 seeds, an array (3, 12)."""
    return np.array([s["test"]["em"] for s in stored(att, H, steps)["seeds"]])


def em12(att, H=64, steps=6000):
    return span(em_seeds(att, H, steps)[:, -1])


fig, ax = plt.subplots(figsize=(8, 3.6))
for att, steps, color, marker, ls, label in (
    ("add", 6000, BLUE, "o", "-", "with attention, 6,000 steps"),
    ("none", 24000, GREEN, "^", "--", "no attention, 24,000 steps"),
    ("none", 6000, ORANGE, "s", "-", "no attention, 6,000 steps"),
):
    em = 100 * em_seeds(att, steps=steps)
    ax.fill_between(ds, em.min(0), em.max(0), color=color, alpha=0.2)
    ax.plot(ds, np.median(em, 0), marker=marker, ls=ls, color=color, label=label)
ax.set(xlabel="number of digits d", ylabel="whole number read correctly (%)", xticks=ds, ylim=(60, 101))
ax.set_title("H = 64, 3 seeds: the bottleneck only shows on long numbers")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()

print("read correctly at 12 digits (3 seeds), by state size H:")
print(f"{'H':>4} {'params no / with att.':>24} {'no att. 6,000 steps':>22} {'24,000 steps':>15} {'att. 6,000':>15}")
for H in (32, 64, 128):
    n_par = f"{num(stored('none', H)['params'], 0)} / {num(stored('add', H)['params'], 0)}"
    print(f"{H:>4} {n_par:>24} {em12('none', H):>22} {em12('none', H, 24000):>15} {em12('add', H):>15}")

- After 6,000 steps, attention with a state of only 32 dimensions (15,371 parameters) reads 99.67–100.00% of the
  12-digit numbers correctly, more than the 128-dimensional fixed vector (163,723 parameters, 94.33–96.67%).
- But train for 4 times longer, and the 64-dimensional fixed vector rises to 98.33–98.67%, the 128-dimensional one
  to 100.00%. The bottleneck makes learning **slower and harder**; it is not a wall: 64 numbers are enough to hold a
  12-digit number, it is just hard to learn how to cram it in.
- With a smaller vector, the bottleneck really is tight: 32 dimensions, even with 24,000 steps, reach only
  72.00–79.67%.

This matches Sutskever et al. (2014): their large LSTM, without attention, “did not have difficulty on long
sentences” (they reversed the source sentence to make learning easier).

Attention makes the problem far *easier to learn*. The next section looks at how it does that.

## 4 · Looking back at the source sentence: one attention step by hand

Bahdanau, Cho & Bengio (2014) let the decoder look back at **every** encoder state $h_1, \dots, h_S$ at each step
$i$, instead of only $h_T$. Three steps, in the row convention:

$$e_{ij} = \tanh(s_{i-1} W_a + h_j U_a)\, v_a, \qquad \alpha_{ij} = \frac{\exp e_{ij}}{\sum_{j'} \exp e_{ij'}}, \qquad c_i = \sum_j \alpha_{ij}\, h_j$$

- The **Query** is $s_{i-1}$, the decoder state before it writes digit $i$: “what am I looking for”.
- The **Key** of position $j$ is $h_j U_a$; a small network with one hidden layer scores $e_{ij}$, how well the query
  and the key match.
- **Softmax** (Lesson 04) turns the scores into **attention weights** $\alpha_{ij}$: positive, summing to 1 over $j$.
- The **Value** is $h_j$ itself; the context $c_i$ is the weighted average of the values. In Bahdanau's model, the
  key and the value come from the same vector $h_j$.

Every operation is differentiable, so the matrices $W_a, U_a, v_a$ are learned with gradients, together with the
rest of the network. The cell below computes the first step when reading “hai nghìn không trăm hai mươi sáu” (2026),
with a loop over the 7 words.

In [ ]:
p = MODELS["add"]
words_a = read(2026)
Xa = np.array([[W2I[w] for w in words_a]])
Ma = np.ones(Xa.shape)
Hs_a, s0 = encode(p, Xa, Ma)  # 7 states h_j (64 numbers each); the decoder starts from s_0 = h_T

query = s0[0] @ p["Wa"]  # the query's part, shared by every j
e = np.zeros(len(words_a))
for j in range(len(words_a)):
    key = Hs_a[0, j] @ p["Ua"]  # the key of position j
    e[j] = np.tanh(query + key) @ p["va"]  # the match score e_1j
alpha = np.exp(e - e.max())  # subtracting the max doesn't change the result, it only avoids overflow
alpha /= alpha.sum()
context = alpha @ Hs_a[0]  # value = h_j

for w, ej, aj in zip(words_a, e, alpha):
    print(f"{w:>6}: e = {num(ej, 2):>6}   α = {pct(aj):>6}  {'█' * round(40 * aj)}")
print(f"sum of the α = {num(alpha.sum(), 4)}; the context c₁ has {context.size} numbers")
vec = softmax_masked(additive_scores(p, s0, Hs_a @ p["Ua"]), Ma)[0]
assert np.allclose(alpha, vec)  # the loop = the vectorized version in decode()
assert abs(e[0] - 13.0327) < 1e-3 and np.allclose(alpha[:2], [0.7129, 0.2744], atol=5e-4), "differs from the reel"

To write the first digit, the decoder puts 71.29% of the weight on “hai” (two) and 27.44% on “nghìn” (thousand):
exactly the two words that decide that the first digit is 2 and that the number has four digits. Scores $e$ a few
units apart are enough: softmax exponentiates, so a score 1 unit higher means a weight $\exp(1) \approx 2.7$ times
larger.

## 5 · The attention map between two sentences

Repeat that step for every digit and we get an **attention map**: each row is a writing step, each column a source
word, and the bluer the cell, the larger the weight; the number in the cell is the weight (%). Each row sums to 1.

In [ ]:
def show_map(words, out, A, title, min_label=0.005):
    """Attention map: rows = writing steps (the digit written), columns = source words; writes the % in cells
    ≥ min_label."""
    labels = [("⟨end⟩" if v == EOS else str(v)) for v in out]
    _, ax = plt.subplots(figsize=(min(10, max(4, 0.42 * len(words) + 1.5)), 0.36 * len(labels) + 1.6))
    ax.imshow(A, cmap=WEIGHTS, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(words)), labels=words, rotation=60, ha="right")
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set(xlabel="source word", ylabel="digit written", title=title)
    for i, row in enumerate(A):
        for j, v in enumerate(row):
            if v >= min_label:
                ax.text(
                    j, i, f"{100 * v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 0.6 else "black"
                )
    return ax


out_a, A_a = run("add", Xa, Ma, 14)
n_out = len(digits_of(out_a[0])) + 1  # the digits and ⟨end⟩
show_map(words_a, out_a[0, :n_out], A_a[0, :n_out], "2026: attention weights (%)")
plt.show()
top = A_a[0, :n_out].argmax(1)
print("heaviest word at each step:", [words_a[j] for j in top], " · read as:", digits_of(out_a[0]))
assert digits_of(out_a[0]) == "2026" and top.tolist() == [0, 2, 4, 6, 6], "differs from the reel"
assert np.allclose(A_a[0].sum(1), 1)

Each digit “looks” at its own place in the sentence: 2 at “hai” (two), 0 at “không trăm” (zero hundred), 2 at
“hai (mươi)” (the two of twenty), 6 at “sáu” (six). Nobody taught the model to align words with digits; this map
emerges on its own because it helps reduce the loss. Bahdanau et al. call this learning to align and translate
jointly.

### 🤔 Predict first
For the long number 964,131,195,884, the source sentence begins “chín trăm sáu mươi **tư** **tỷ** một trăm…” (nine
hundred sixty-**four** **billion** one hundred…). When it writes the third digit (the 4, which comes from “tư”), which
word does the decoder put the most weight on?

In [ ]:
out_b, A_b = run("add", Xb, Mb, 14)
n_b = len(digits_of(out_b[0])) + 1
show_map(
    words_b, out_b[0, :n_b], A_b[0, :n_b], "964,131,195,884: attention weights (%), cells ≥ 5% only", min_label=0.05
)
plt.show()
for i in (2, 6):  # step 3 writes the digit 4 (of “tư”), step 7 the digit 1 (the start of “một trăm chín mươi lăm”)
    a_i = A_b[0, i]
    top3 = " · ".join(f"“{words_b[j]}” (word {j + 1}) {pct(a_i[j])}" for j in np.argsort(-a_i)[:3])
    print(f"step {i + 1} (writes {out_b[0, i]}): {top3}")
print("heaviest word at each step:", " · ".join(words_b[j] for j in A_b[0, :n_b].argmax(1)))
assert A_b[0, :n_b].argmax(1).tolist() == [0, 2, 5, 6, 8, 10, 11, 14, 16, 18, 20, 22, 22], "differs from the reel"

Not “tư”, but “tỷ”, the word *right after* it. The reason: $h_j$ is not the $j$-th word, but the GRU state **after
reading** words $1 \dots j$. The state at “tỷ” has just finished reading “…sáu mươi tư tỷ” (…sixty-four billion), so
it carries both the digit 4 and the position of the group (the billions). The model learns to look at the states that
are most convenient *for it*, not necessarily the words a human reader would point to. At step 7 (the digit 1 that
opens “một trăm chín mươi lăm nghìn”, one hundred ninety-five thousand), the weight is split almost evenly between
“triệu” (million), the word before it, and “một” (one).

(Bahdanau et al. use a bidirectional RNN, so their $h_j$ summarizes the words both before and after $j$. The encoder
here reads only from left to right, so the state at a word knows only the words up to it.)

So the attention map shows **where the model looks**, but reading it as “this digit comes from that word” easily
goes wrong. The next section tests that with direct interventions.

## 6 · Do the attention weights explain the prediction?

Jain & Wallace (2019) argue that attention is **not** an explanation (on classification, question-answering and
natural language inference tasks): the weights often disagree with other measures of importance, and very different
maps still give the same prediction. Wiegreffe & Pinter (2019) counter that this depends on how “explanation” is
defined, and propose stricter tests. The number-reading task lets us try a few interventions right away, without
needing the debate: change the input or change the weights, then look at the output.

In [ ]:
w_ba = list(words_b)
w_ba[4] = "ba"  # "…sáu mươi tư tỷ…" → "…sáu mươi ba tỷ…" (sixty-four billion → sixty-three billion)
X_ba = np.array([[W2I[w] for w in w_ba]])
print("change “tư” → “ba”:", num(int(digits_of(run("add", X_ba, Mb, 14)[0][0])), 0))


def drop_word(j):
    """Set the weight of position j to 0 at every step, then renormalize the remaining weights."""

    def edit(a, M):
        a = a.copy()
        a[:, j] = 0
        return a / a.sum(1, keepdims=True)

    return edit


for j in (4, 5):
    print(f"never look at “{words_b[j]}”:", num(int(digits_of(run("add", Xb, Mb, 14, edit=drop_word(j))[0][0])), 0))
assert digits_of(run("add", X_ba, Mb, 14)[0][0]) == "963131195884"
assert digits_of(run("add", Xb, Mb, 14, edit=drop_word(4))[0][0]) == str(NB)

Change “tư” to “ba” (three) and the third digit becomes 3: the prediction **depends** on “tư”, even though “tư” gets
only 10.71% of the weight. Yet forbid the decoder to look at “tư” at all, and it still reads the whole number
correctly, because the information of “tư” is already in the state at “tỷ”. A low weight does not mean unimportant.

### 🤔 Predict first
Replace the attention weights with a **uniform** average (every word the same) at every step. On the 300 12-digit
numbers, what percentage does the model still read correctly? And if it keeps only the **one** heaviest word at each
step (“hard” attention)?

In [ ]:
def uniform(a, M):
    return M / M.sum(1, keepdims=True)


def top_k(k):
    """Keep only the k heaviest positions at each step, renormalized."""

    def edit(a, M):
        keep = np.argsort(-a, 1)[:, :k]
        b = np.zeros_like(a)
        np.put_along_axis(b, keep, np.take_along_axis(a, keep, 1), 1)
        return b / b.sum(1, keepdims=True)

    return edit


edits = {"normal": None, "uniform": uniform, "heaviest word only": top_k(1), "2 heaviest words": top_k(2)}
acc_edit = {name: exact("add", 12, f).mean() for name, f in edits.items()}
for name, v in acc_edit.items():
    print(f"{name:>20}: reads {pct(v)} of the 12-digit numbers correctly")
assert acc_edit["normal"] == 1 and acc_edit["uniform"] == 0
assert round(N_TEST * acc_edit["heaviest word only"]) == 80 and round(N_TEST * acc_edit["2 heaviest words"]) == 295

Three results, each teaching something:
- **Uniform → 0%.** The model really *uses* the attention weights: take them away and it can't read a single 12-digit
  number. This is the “uniform weights” test that Wiegreffe & Pinter propose. They found that on three of the
  classification tasks Jain & Wallace used, uniform weights hardly made the results worse, so those tasks can't be
  used to ask whether attention explains anything.
- **Only 1 word → about a quarter.** The information of a digit is often spread over two adjacent states (like “tư”
  and “tỷ” above), so “the heaviest word” is not a complete explanation.
- **2 words → almost enough.** Most of the remaining weight is only small noise.

Conclusion: attention weights show **where** the model looks, but to know **why** it predicts what it does, you have
to intervene (change the input, edit the weights) and measure again, as we just did.

## 7 · The dot product, and why divide by $\sqrt{d_k}$

Luong, Pham & Manning (2015) score more compactly, with a dot product between the decoder state and $h_j$, with no
$W_a, U_a, v_a$ (they use the state of the current step; the course's dot-product models use $s_{i-1}$, as Bahdanau
does). The Transformer (Vaswani et al., 2017) uses $q \cdot k / \sqrt{d_k}$, written for the whole sentence at once:

$$\operatorname{Attention}(Q, K, V) = AV, \qquad A = \operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)$$

Each row of $Q, K, V$ is a token; the softmax is taken row by row. The reason for dividing: if the components of $q$
and $k$ are independent, with mean 0 and variance 1, then $q \cdot k = \sum_{t=1}^{d_k} q_t k_t$ is a sum of $d_k$
terms of variance 1, so it has variance $d_k$. The authors argue that when $d_k$ is large, large scores push the
softmax into a region where it is nearly one-hot, where the gradients are very small.

The cell below draws $q$ and 8 keys with $d_k = 64$ (64 is the $d_k$ of the original Transformer) with exactly the
demo's random number generator (mulberry32 + Box–Muller, seed 1), so that the printed numbers match the
“Why divide by √dₖ” card on the lesson page.

In [ ]:
def mulberry32(seed, n):
    """n uniform numbers in [0, 1) from mulberry32(seed), identical to the demo's JavaScript function."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def gaussian(seed, n):
    """n N(0, 1) numbers by Box–Muller, using both the cos and the sin of each pair, as the demo does."""
    u = mulberry32(seed, n + n % 2)
    radius, theta = np.sqrt(-2 * np.log(1 - u[0::2])), 2 * np.pi * u[1::2]
    return np.stack([radius * np.cos(theta), radius * np.sin(theta)], 1).ravel()[:n]


def softmax(z):
    a = np.exp(z - z.max(-1, keepdims=True))
    return a / a.sum(-1, keepdims=True)


dk = 64
g = gaussian(1 * 104729 + dk, dk + 8 * dk)  # the demo's seed numbering: seed · 104729 + dₖ
q, K = g[:dk], g[dk:].reshape(8, dk)
scores = K @ q
raw, scaled = softmax(scores), softmax(scores / np.sqrt(dk))
print("q·k of the 8 keys:", " ".join(num(v, 1) for v in scores))
print("not divided:   ", " ".join(pct(v, 1) for v in raw))
print("divided by √64:", " ".join(pct(v, 1) for v in scaled))
assert abs(scores[3] - 16.2747) < 1e-3 and abs(raw.max() - 0.9974) < 1e-4 and abs(scaled.max() - 0.4821) < 1e-4

Without the division, the fourth key gets 99.7% of the weight: the softmax almost makes a hard choice of one key.
Divided by $\sqrt{64} = 8$, the same scores give weights spread over several keys. One draw says nothing yet; the cell
below repeats it 500 times for each $d_k$ from 1 to 1,024 (the demo's same way of drawing, seed 2026) and measures
three things:
- the variance of $q \cdot k$ (in theory: $d_k$);
- the mean largest weight, and the fraction of draws in which one key gets more than 99%;
- the size of the softmax's gradient $\lVert \partial a / \partial z \rVert$ (the Frobenius norm of the Jacobian
  matrix $\operatorname{diag}(a) - a^{\top} a$): how small the gradient flowing back through the softmax gets.

In [ ]:
DKS, DRAWS, DMAX = [1, 4, 16, 64, 256, 1024], 500, 1024
G = gaussian(2026, DRAWS * 9 * DMAX).reshape(DRAWS, 9 * DMAX)  # each draw: q (1,024 numbers), then 8 keys × 1,024


def jacobian_norm(a):
    """‖diag(a) − aᵀa‖ of the softmax, for each row of a."""
    J = a[:, :, None] * np.eye(a.shape[1]) - a[:, :, None] * a[:, None, :]
    return np.sqrt((J**2).sum((1, 2)))


stats = {}
for d in DKS:
    q_, K_ = G[:, :d], G[:, DMAX:].reshape(DRAWS, 8, DMAX)[:, :, :d]
    e_ = np.einsum("bkt,bt->bk", K_, q_)  # 500 × 8 scores
    row = {"var": (e_**2).mean()}
    for name, z in (("raw", e_), ("scaled", e_ / np.sqrt(d))):
        a = softmax(z)
        row[name] = (a.max(1).mean(), (a.max(1) > 0.99).mean(), jacobian_norm(a).mean())
    stats[d] = row
    print(
        f"dₖ = {num(d, 0):>5}: var(q·k) = {num(row['var'], 1):>7} · largest {num(row['raw'][0], 3)} → "
        f"{num(row['scaled'][0], 3)} · >99%: {pct(row['raw'][1], 1):>6} → {pct(row['scaled'][1], 1)}"
        f" · ‖∂a/∂z‖ {num(row['raw'][2], 3)} → {num(row['scaled'][2], 3)}"
    )
s64 = stats[64]
assert abs(s64["var"] - 63.7691) < 1e-3 and abs(s64["raw"][0] - 0.8809) < 1e-4 and abs(s64["scaled"][0] - 0.3698) < 1e-4
assert s64["raw"][1] == 0.402 and s64["scaled"][1] == 0, "differs from the demo"

The figure below plots the largest weight and $\lVert \partial a / \partial z \rVert$ against $d_k$ (log horizontal
axis): not divided (orange, squares) and divided by $\sqrt{d_k}$ (blue, round dots).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
for key, color, marker, label in (("raw", ORANGE, "s", "not divided"), ("scaled", BLUE, "o", "divided by √dₖ")):
    a1.plot(DKS, [stats[d][key][0] for d in DKS], marker=marker, color=color, label=label)
    a2.plot(DKS, [stats[d][key][2] for d in DKS], marker=marker, color=color, label=label)
a1.set(
    xscale="log",
    ylim=(0, 1),
    xlabel="dₖ",
    ylabel="largest weight (mean)",
    title="Without division, softmax gets peaked",
)
a2.set(
    xscale="log",
    ylim=(0, 0.45),
    xlabel="dₖ",
    ylabel="‖∂a/∂z‖ (mean)",
    title="…and the gradient through it shrinks",
)
a1.legend(frameon=False)
for ax in (a1, a2):
    ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()

Without the division, the largest weight approaches 1 as $d_k$ grows, and the gradient flowing back through the
softmax shrinks: at $d_k = 1{,}024$ it is less than an eighth of the divided version's. Divided by $\sqrt{d_k}$, both
stay almost constant with $d_k$. At $d_k = 1$ the two are the same.

### The “variance 1” assumption on a real model
The argument above assumes that each component of $q$ and $k$ has variance 1. GRU states are different: they go
through $\tanh$ and the gates, so they are much smaller. The cell below measures on the learned additive model (300
12-digit numbers): the variance of the components of $h_j$; then it tries Transformer-style scoring
$e_j = h_T \cdot h_j / \sqrt{64}$ on those same states, and measures the **entropy** of the weights (in bits; uniform
weights over $S$ words give $\log_2 S$ bits, all of it on one word gives 0 bits).

In [ ]:
def entropy_bits(A):
    """Entropy (bits) of each weight distribution along the last axis; 0·log 0 = 0."""
    return -np.where(A > 0, A * np.log2(np.maximum(A, 1e-300)), 0).sum(-1)


Hs12, hT12 = encode(MODELS["add"], X12, M12)
var_h = Hs12[M12 > 0].var()
e_dot = np.einsum("bsh,bh->bs", Hs12, hT12) / np.sqrt(64)
uniform_bits = np.log2(M12.sum(1)).mean()
_, A12 = run("add", X12, M12, 13)
ent_add = entropy_bits(A12).mean()
print(f"variance of the components of hⱼ: {num(var_h, 3)} (the √dₖ assumption: 1)")
print(f"scores h_T·hⱼ/8: standard deviation {num(e_dot[M12 > 0].std(), 2)}")
print(f"entropy: Transformer-style scoring  {num(entropy_bits(softmax_masked(e_dot, M12)).mean(), 2)} bits")
print(f"         learned additive attention {num(ent_add, 2)} bits   (fully uniform: {num(uniform_bits, 2)} bits)")
add_att = stored("add")["seeds"][0]["att"]
assert abs(var_h - add_att["var"]) < 1e-4 and abs(ent_add - add_att["entropy"]) < 1e-4, "differs from the demo"

The components of $h_j$ have a variance of about 0.14, not 1, so $h_T \cdot h_j / 8$ gives very small scores and an
almost uniform softmax (3.77 bits, close to the fully uniform 4.08 bits). The learned additive model is far more
focused (1.45 bits).

The course's sweep did train two dot-product models on GRU states, H = 64: divided by $\sqrt{64}$ (`dot`) and not
divided (`dotu`, as in Luong). The cell below reads their results from the sweep file: the entropy of the attention
and the fraction read correctly at 12 digits, for all 3 seeds, together with the variance of the components of each
model's own GRU states $h_j$.

In [ ]:
rows = [
    ("no attention", "none", 6000),
    ("additive", "add", 6000),
    ("divided by √64", "dot", 6000),
    ("divided by √64, 24,000 steps", "dot", 24000),
    ("not divided", "dotu", 6000),
]
print(f"{'':>28}  {'entropy (bits)':>14}  {'variance of hⱼ':>14}  {'correct at 12 digits':>20}")
for label, att, steps in rows:
    seeds = stored(att, steps=steps)["seeds"]
    ent = span([s["att"]["entropy"] for s in seeds], lambda v: num(v, 2)) if "att" in seeds[0] else "—"
    var = span([s["att"]["var"] for s in seeds], lambda v: num(v, 2)) if "att" in seeds[0] else "—"
    print(f"{label:>28}  {ent:>14}  {var:>14}  {span([s['test']['em'][-1] for s in seeds]):>20}")
print(
    f"(fully uniform: {num(stored('dot')['seeds'][0]['att']['uniform'], 2)} bits; variance of hⱼ at initialization: "
    f"{span([s['init']['var'] for s in stored('dot')['seeds']], lambda v: num(v, 3))})"
)

This is a result worth remembering:
- **Divided by $\sqrt{64}$ on GRU states:** the attention is almost uniform (4.05–4.06 bits, against 4.08 bits if
  fully uniform), and a uniform average of the $h_j$ is again a fixed vector, which is almost back to the bottleneck.
  The model still reaches 77.67–83.67% at 12 digits, closer to the no-attention level than to the additive one;
  trained 4 times longer, it rises to 98.33–99.33%, but the attention is **still** nearly uniform: it learns to read
  without looking at the right place.
- **Not divided:** 100.00% for every seed, with focused attention (0.94–1.05 bits).

So $\sqrt{d_k}$ is not magic: it is the right scale when the components have a variance near 1. In the model divided
by $\sqrt{64}$, the GRU states are even smaller (variance 0.07–0.08); they lie in $(-1, 1)$ and there is no matrix
to relearn the scale, so the extra division by 8 makes the scores too flat. In the Transformer, $q$ and $k$ are the
learned projections $xW_q$, $xW_k$ of vectors that have just been normalized (Lesson 10), so the model can adjust the
size of the scores itself. (Bahdanau's model doesn't run into this: $v_a$ and $W_a, U_a$ learn the scale of the
scores themselves.)

## 8 · Self-attention: a sentence attending to itself

In **self-attention**, the queries, keys and values are all computed from the same sequence. The encoder here
replaces the GRU with exactly one such layer (`reel/attention/s2s_sa.py`), with $X_0$ the word embeddings (one row
per word):

$$Q = X_0 W_q,\quad K = X_0 W_k,\quad V = X_0 W_v,\qquad H = \tanh\!\left(X_0 + \operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V\right)$$

There is no loop reading from left to right: each word connects directly to every word, including itself, in a
single step. The decoder is still a GRU with additive attention, starting from the mean of the rows of $H$. There are
two versions: `sa` has nothing about position, and `sapos` adds a learned position vector $P_j$ to the $j$-th word
(like GPT-2's `wpe`).

In [ ]:
def encode_self(p, X, M):
    """The self-attention encoder. Returns Hs (B, S, H), the decoder's initial state s₀ and the attention matrix A
    (B, S, S)."""
    X0 = p["Ex"][X] + (p["Pe"][: X.shape[1]][None] if "Pe" in p else 0)  # only sapos has position vectors
    Q, K, V = X0 @ p["Wq"], X0 @ p["Wk"], X0 @ p["Wv"]
    A = softmax_masked(np.einsum("bih,bjh->bij", Q, K) / np.sqrt(Q.shape[-1]), M[:, None, :])  # each row sums to 1
    Hs = np.tanh(X0 + A @ V)
    s0 = (Hs * M[:, :, None]).sum(1) / M.sum(1, keepdims=True)  # the mean over the words: independent of the order
    return Hs, s0, A


def run_self(name, X, M, steps):
    p = MODELS[name]
    Hs, s0, A = encode_self(p, X, M)
    return decode(p, Hs, s0, M, steps, "add")[0], A


def ids(words):
    return np.array([[W2I[w] for w in words]]), np.ones((1, len(words)))


for name in ("sa", "sapos"):
    ok = [int((run_self(name, *TEST[d][:2], D_MAX + 1)[0][:, : d + 1] == TEST[d][2]).all(1).sum()) for d in TEST]
    print(f"{name:>5}: correct out of 300 at d = 1…12:", ok, f"· total {num(sum(ok), 0)}/{num(D_MAX * N_TEST, 0)}")
    assert ok == seed0_counts(name), "differs from the demo: check the model"

With position vectors, the model reads all 3,600 test numbers correctly. Without them, it gets everything right only
for 1-digit numbers; from 2 digits on, the fraction correct drops fast, and at 12 digits only 1 number in 300 is
left.

### 🤔 Predict first
Give “hai mươi ba” (23) and “ba mươi hai” (32) to the model **without** positions. Does it read two different numbers,
or one?

In [ ]:
out_sa, out_pos = {}, {}
for n in (23, 32):
    w = read(n)
    out_sa[n] = digits_of(run_self("sa", *ids(w), 4)[0][0])
    out_pos[n] = digits_of(run_self("sapos", *ids(w), 4)[0][0])
    print(f"“{' '.join(w)}” → without positions: {out_sa[n]}   with positions: {out_pos[n]}")
assert out_sa == {23: "23", 32: "23"} and out_pos == {23: "23", 32: "32"}, "differs from the reel and the demo"

The same answer, 23, for two different sentences. The reason is right there in the formula: without $P_j$, row $i$
of $H$ depends only on the $i$-th *word* and on the *set* of words, not on where they stand. Shuffle the words, and
the rows of $Q, K, V$ are permuted with them, the matrix $A$ has its rows and its columns permuted in the same way,
and $H$ only has its rows permuted: self-attention is **permutation-equivariant**. After that, the initial state is
the mean of the rows, and the decoder's attention is a weighted sum over all positions, so the output is
**invariant** to every ordering. The cell below checks both:
- the attention matrix of “ba mươi hai” is exactly the matrix of “hai mươi ba” with its rows and columns permuted;
- all 6 orderings of the three words, and all 5,040 orderings of the 7 words of 2026, give the same output.

In [ ]:
w23, w32 = read(23), read(32)
perm = [w23.index(w) for w in w32]  # the position in "hai mươi ba" of each word of "ba mươi hai"
A23, A32 = run_self("sa", *ids(w23), 4)[1][0], run_self("sa", *ids(w32), 4)[1][0]
print("attention matrix of “hai mươi ba” (rows: the querying word, columns: the word looked at):")
for w, row in zip(w23, A23):
    print(f"{w:>5}:", "  ".join(num(v, 3) for v in row))
gap = np.abs(A32 - A23[perm][:, perm]).max()
print("largest difference between A(ba mươi hai) and the permuted A(hai mươi ba):", "0" if gap == 0 else num(gap, 15))
assert gap < 1e-12


def all_orders(n, name="sa"):
    """The model's outputs for every ordering of the words that read the number n."""
    w = read(n)
    X = np.array([[W2I[w[k]] for k in order] for order in itertools.permutations(range(len(w)))])
    out, _ = run_self(name, X, np.ones(X.shape), len(str(n)) + 2)
    return len(X), {digits_of(r) for r in out}


for n in (23, 105, 2026):
    count, outs = all_orders(n)
    print(f"{' '.join(read(n))}: {num(count, 0)} orderings → distinct outputs: {sorted(outs)}")
assert all_orders(23)[1] == {"23"} and all_orders(105)[1] == {"501"} and all_orders(2026) == (5040, {"2062"})

The difference is exactly 0: the permutation only rearranges the same numbers. The model without positions reads
every ordering of “một trăm linh năm” (105) as 501, and every ordering of 2026 as 2062: it sees only a *bag of
words*, and for each bag it has to settle on one number. No model that sees only the bag of words can do better than
always choosing the **most frequent number** among the numbers with the same bag of words. The cell below computes
that ceiling exactly for $d \le 5$, by enumerating every number and its probability under the sampler.

In [ ]:
def prob(n):
    """The probability of the number n under sample(), given its number of digits."""
    s = str(n)
    p_digit = lambda c: P_ZERO if c == "0" else (1 - P_ZERO) / 9
    lead = s[: len(s) - 3 * ((len(s) - 1) // 3)]  # the first group (1–3 digits); the first digit is uniform on 1–9
    p = 1 / 9 * math.prod(p_digit(c) for c in lead[1:])
    for g0 in range(len(lead), len(s), 3):
        grp = s[g0 : g0 + 3]
        p *= (1 - P_GROUP) * math.prod(p_digit(c) for c in grp) + (P_GROUP if grp == "000" else 0)
    return p


def bag_ceiling(d):
    """The best fraction correct of any reader that sees only the bag of words: for each bag, always answer the most
    probable number."""
    best = defaultdict(float)
    for n in range(10 ** (d - 1), 10**d):
        bag = tuple(sorted(read(n)))
        best[bag] = max(best[bag], prob(n))
    return sum(best.values())  # the probabilities of all the d-digit numbers sum to 1


ceiling = [bag_ceiling(d) for d in range(1, 6)]
ok_sa = [int(v) for v in seed0_counts("sa")]
for d, c in enumerate(ceiling, 1):
    got = ok_sa[d - 1] / N_TEST
    print(
        f"d = {d}: ceiling {pct(c)} · model without positions (seed 0) {pct(got)} · "
        f"{num(100 * (c - got), 2)} points below the ceiling"
    )
assert np.allclose(ceiling, sweep["ceiling"][:5], atol=5e-5), "differs from the demo"
assert all(c - n_ok / N_TEST < 1 - c for c, n_ok in zip(ceiling[1:], ok_sa[1:]))  # most misreads lie below the ceiling

The model without positions stays a few points below the ceiling at every length we can compute (at most 4.51 points
below; at 3 digits: 54.00% against a ceiling of 54.71%): most of its misreads come from missing information, because
all it sees is the bag of words. It has learned close to the best it can *with what it can see*. The figure below
plots the whole sweep, 3 seeds per model (line: median, band:
lowest–highest): with positions (blue, round dots), without positions (orange, squares), and the ceiling of every
bag-of-words reader (gray, x marks; the sweep file also has $d = 6$).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for att, color, marker, label in (("sapos", BLUE, "o", "with positions"), ("sa", ORANGE, "s", "without positions")):
    em = 100 * em_seeds(att)
    ax.fill_between(ds, em.min(0), em.max(0), color=color, alpha=0.2)
    ax.plot(ds, np.median(em, 0), marker=marker, color=color, label=label)
ceil_all = sweep["ceiling"]
ax.plot(
    range(1, len(ceil_all) + 1),
    100 * np.array(ceil_all),
    "x--",
    color=GREY,
    ms=8,
    label="ceiling of every bag-of-words reader",
)
ax.set(xlabel="number of digits d", ylabel="whole number read correctly (%)", xticks=ds, ylim=(-2, 102))
ax.set_title("Self-attention encoder, 3 seeds: no positions, no word order")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()
print(f"with positions, every length and every seed: {span(em_seeds('sapos').ravel())}")
print(f"without positions, 12 digits: {em12('sa')} · ceiling at d = 6: {pct(ceil_all[5])}")

With position vectors, all 3 seeds read 100.00% correctly at every length: one self-attention layer plus positions is
enough for this task. Without them, every seed stops on almost the same line, a few points below the ceiling: the
limit is mainly information, not learning. That is why every Transformer has to add position somehow: GPT-2 adds a
learned position vector to each token (Lesson 17), and newer models rotate $q$ and $k$ according to position (RoPE).

### A shortcut for the gradient
Attention also helps learning in another way. Without attention, the error signal of the first digit has to flow back
through all 22 GRU steps to reach the first word. With attention, every $h_j$ connects directly to the decoder, like
the skip connections of [Lesson 13](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/). The sweep measured
$\lVert \partial L_1 / \partial x_j \rVert$ at initialization (the error at the first digit, with respect to the
input of each word), averaged over 64 12-digit numbers read with exactly 23 words, then divided by the value at the
last word:

In [ ]:
def sci(v):
    """A positive number in the form 3.1·10⁻⁴."""
    e = math.floor(math.log10(v))
    return f"{num(v / 10**e, 1)}·10{str(e).translate(SUP)}" if not -1 <= e <= 2 else num(v, 2)


g_none = [s["gradInit"][0] for s in stored("none")["seeds"]]  # log10 ‖∂L₁/∂x₁‖ / ‖∂L₁/∂x₂₃‖
g_add = [s["gradInit"][0] for s in stored("add")["seeds"]]
print(f"the first word gets (relative to the last word): no attention   {span([10**v for v in g_none], sci)}")
print(f"                                                 with attention {span([10**v for v in g_add], sci)}")
print(f"{span([10 ** (a - n) for a, n in zip(g_add, g_none)], lambda v: num(v, 0))} times more (seed by seed)")

At initialization, without attention, the first word gets only a few ten-thousandths of the last word's gradient: the
signal shrinks at every GRU step, as in Lesson 09 and Lesson 15. With attention, it gets about a fifth, a few hundred
times more. The model without attention can still learn to carry the signal across 22 steps with the GRU's gates
(section 3 shows it rising to 98.33–98.67% when trained long enough), but attention has that shortcut from the start,
so it learns faster.

## 9 · Try changing things

The cells below run the same functions again with other parameters. You can edit the values and run them again.

### Read any number
Change the numbers in the list. Reading the whole test set correctly doesn't mean reading **every** number correctly:
the test set is drawn from the same distribution as the training data, while “odd” numbers such as 12 identical
digits are almost never drawn.

In [ ]:
for n in [2026, 15, 1000005, 100000000000, 101010101010, 999999999999]:
    X, M = ids(read(n))
    got = {name: digits_of(run(name, X, M, 16)[0][0]) for name in ("none", "add")}
    marks = {k: "right" if v == str(n) else "WRONG" for k, v in got.items()}
    print(f"{num(n, 0):>17}: no att. {got['none']:>16} ({marks['none']}) · with att. {got['add']:>16} ({marks['add']})")
assert digits_of(run("add", *ids(read(999999999999)), 16)[0][0]) == "99999999999"

The model with attention reads 999,999,999,999 with one 9 missing, perhaps because with 12 identical digits the
states $h_j$ are very similar, and it loses track of which group it is in. The model without attention, for its part,
reads “một trăm tỷ” (one hundred billion) as 13 digits.

### The sharpness of the softmax
Multiply the scores $e_{ij}$ of the learned additive model by a factor $k$ before the softmax ($k = 1$ is normal,
$k = 0$ gives uniform weights, $k \to \infty$ keeps only one word). This is the same question as $\sqrt{d_k}$ in
section 7: the scale of the scores decides whether the softmax is flat or peaked.

In [ ]:
def sharpen(k):
    """Edit the weights by multiplying the scores by k: softmax(k·e) = a^k / Σ a^k."""

    def edit(a, M):
        if k == 0:
            return M / M.sum(1, keepdims=True)
        if np.isinf(k):
            return np.eye(a.shape[1])[a.argmax(1)]
        b = np.where(M > 0, np.exp(k * np.log(np.maximum(a, 1e-300))), 0)
        return b / b.sum(1, keepdims=True)

    return edit


for k in [0, 0.25, 0.5, 1, 2, 4, np.inf]:
    label = "∞" if np.isinf(k) else num(k, 2).rstrip("0").rstrip(".")
    print(f"k = {label:>4}: reads {pct(exact('add', 12, sharpen(k)).mean()):>7} of the 12-digit numbers correctly")

Too flat ($k \le 0.5$) or too peaked ($k \ge 4$), either way the model reads worse; it was trained with exactly
$k = 1$, and that scale is part of what it learned.

### A different test set
Draw a new 12-digit set (change `seed`) and compare the two models. With 300 numbers, differences of a few percentage
points between sets are normal.

In [ ]:
for seed in (3000, 3001, 3002):
    X, M, Y = encode_set(sample(np.random.default_rng(seed), 12, N_TEST))
    acc = {name: (run(name, X, M, 13)[0] == Y).all(1).mean() for name in ("none", "add")}
    print(f"seed {seed}: no attention {pct(acc['none'])} · with attention {pct(acc['add'])}")

## 10 · Limits and next steps

- **There is still a loop.** The GRU encoder and decoder read and write one step at a time, so they can't be computed
  in parallel along the sequence. The Transformer (Vaswani et al., 2017) drops the RNN entirely: every layer is
  self-attention plus a small MLP.
- **One head, one layer.** The self-attention encoder of section 8 has only one attention matrix. The Transformer runs
  several “heads” in parallel (multi-head attention), each with its own $W_q, W_k, W_v$, and stacks many layers.
- **Quadratic cost.** The matrix $A$ has $S \times S$ entries: a sentence twice as long costs four times as much. For
  a 23-word sentence that's fine; for texts of hundreds of thousands of tokens it is very expensive (Lesson 25 looks
  for ways around it).
- **Text generation.** GPT has only the decoder part, and *masks* the attention matrix so that each token only looks
  back. That mask also breaks the permutation symmetry, so masked multi-layer models can still infer some positional
  information even without position vectors (Haviv et al., 2022); our unmasked encoder can't.

Lesson 17 puts these pieces together into the Transformer and GPT-2, and looks at the 144 attention maps of a real
model.

## Summary
- Forcing the whole sentence into a fixed vector is a bottleneck: H = 64 reads only 72.33–75.00% of the 12-digit
  numbers correctly after 6,000 steps, and the errors are mostly two digits swapped in the middle of the number
  (964,131,195,884 → 964,131,915,884). But it is a bottleneck for *learning*: trained 4 times longer, it rises to
  98.33–98.67%.
- Attention: the score $e_{ij}$ measures how well the query $s_{i-1}$ matches the key $h_j U_a$; softmax turns the
  scores into positive weights that sum to 1; the context is the weighted average of the values $h_j$. The additive
  model reads 100.00% correctly.
- The attention map learns the alignment on its own, but it shows where the model *looks*, not *why*: the digit 4
  looks at “tỷ” (76.45%), yet it is changing “tư” that changes that digit. Remove attention (uniform weights) → 0%;
  keep only 1 word → 26.67%.
- $\operatorname{Var}(q \cdot k) = d_k$ when the components have variance 1; dividing by $\sqrt{d_k}$ keeps the
  softmax from getting too peaked (largest weight 0.881 → 0.370 at $d_k = 64$). On GRU states (variance 0.07–0.08 in
  that very model), the division makes the attention nearly uniform (4.06 bits), and the model reaches only
  77.67–83.67%.
- Self-attention without positions is permutation-equivariant: “ba mươi hai” → 23, every ordering of the 7 words of
  2026 → 2062, close to the ceiling of every bag-of-words reader. Add position vectors: 100.00% at every length.

## Review questions
1. The model without attention misreads 83 of the 12-digit numbers, mostly by swapping two adjacent digits. What does
   that say about what $h_T$ still keeps and what it loses?
2. Write the three formulas of one additive attention step. In Bahdanau's model, what are the key and the value of
   position $j$?
3. When it writes the 4 of 964,131,195,884, the largest weight falls on “tỷ”, not on “tư”. Why? Which test shows that
   the digit really depends on “tư”?
4. If the components of $q, k \in \mathbb{R}^{d_k}$ are independent, with mean 0 and variance 1, what is the variance
   of $q \cdot k$? Why does that make the softmax peaked and the gradient small?
5. On GRU states, dividing by $\sqrt{64}$ makes the attention almost uniform, yet the model still reads 77.67–83.67%
   of the 12-digit numbers correctly. How does it read them? Why does the undivided model do better?
6. Why does the self-attention encoder without positions give the same answer for “hai mươi ba” and “ba mươi hai”? At
   most what percentage of the 3-digit numbers can it get right?

<details><summary>Hints and answers</summary>

1. In 50 of the 83 wrong numbers, every digit is present and only two adjacent digits are swapped: $h_T$ still keeps
   *which digits* the number has, but the order in the middle of the number gets blurred. After 6,000 steps, the model
   hasn't yet finished learning how to cram the positions into the 64 numbers as well.
2. $e_{ij} = \tanh(s_{i-1}W_a + h_jU_a)\,v_a$; $\alpha_{ij} = \operatorname{softmax}_j(e_{ij})$;
   $c_i = \sum_j \alpha_{ij} h_j$. The key is $h_j$ (through $U_a$), and the value is $h_j$ too.
3. $h_j$ is the state *after* the GRU has read the words up to $j$, so the state at “tỷ” already contains “tư”,
   together with the billions place. Changing “tư” to “ba” changes the digit to 3, while forbidding the decoder to
   look at “tư” changes nothing: the information sits in another state.
4. $d_k$. The scores have standard deviation $\sqrt{d_k}$, so the gap between the largest score and the others grows;
   softmax exponentiates, so it puts almost all the weight on one key, and the Jacobian
   $\operatorname{diag}(a) - a^{\top}a$ goes to 0.
5. A nearly uniform average of all the $h_j$ is an almost fixed vector, so it reads like the model without attention:
   by cramming the number into the states. Without the division, the scores are large enough for the softmax to
   focus, so it really looks at the right place (0.94–1.05 bits).
6. Without position vectors, each state depends only on its own word and on the set of words; the initial state is a
   mean, and the decoder's attention is a weighted sum. The two sentences have the same bag of words, so they get the
   same output. The ceiling at 3 digits is 54.71%.
</details>

## Further reading
- I. Sutskever, O. Vinyals, Q. Le (2014), [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215).
- K. Cho et al. (2014), [On the Properties of Neural Machine Translation: Encoder–Decoder Approaches](https://arxiv.org/abs/1409.1259):
  quality drops quickly as sentences get longer.
- D. Bahdanau, K. Cho, Y. Bengio (2014), [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473):
  additive attention; RNNsearch-50 does not degrade even on sentences of 50 words or more.
- M.-T. Luong, H. Pham, C. Manning (2015), [Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025):
  dot-product scores.
- A. Vaswani et al. (2017), [Attention Is All You Need](https://arxiv.org/abs/1706.03762): $\sqrt{d_k}$ (section 3.2.1).
- D. Britz et al. (2017), [Massive Exploration of Neural Machine Translation Architectures](https://arxiv.org/abs/1703.03906):
  compares additive and multiplicative attention.
- S. Jain, B. Wallace (2019), [Attention is not Explanation](https://arxiv.org/abs/1902.10186) and S. Wiegreffe,
  Y. Pinter (2019), [Attention is not not Explanation](https://arxiv.org/abs/1908.04626).
- A. Haviv et al. (2022), [Transformer Language Models without Positional Encodings Still Learn Positional Information](https://arxiv.org/abs/2203.16634).
- d2l.ai, [the chapter Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html):
  query–key–value, Bahdanau attention, self-attention and positional encoding, with code.

**Next lesson:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/):
stacking multi-head self-attention, masking so that each token only looks back, and reading the inside of GPT-2.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")